# ASSIGNMENT GenAI – Prompt Templates using LangChain
## Mastering Prompt Templates in LangChain

**Pipeline Flow:** User Input → Validation → Prompt Template → Dynamic Prompt Generation → Output

## Setup & Installation

In [1]:
# Install required libraries
!pip install langchain langchain-core -quiet


Usage:   
  pip install [options] <requirement specifier> [package-index-options] ...
  pip install [options] -r <requirements file> [package-index-options] ...
  pip install [options] [-e] <vcs project url> ...
  pip install [options] [-e] <local project path> ...
  pip install [options] <archive url/path> ...

no such option: -u


In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

---
## Task 1: Replace Hardcoded Prompts (10%)

**Goal:** Convert the hardcoded f-string function into a reusable LangChain `PromptTemplate`.

In [3]:
task1_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners"
)

# Reusable function using the template
def explain_topic(topic):
    """Generate a beginner-friendly explanation prompt for any topic."""
    return task1_template.format(topic=topic)

# Test the template
print("=== Task 1 Output ===")
print(explain_topic("Machine Learning"))
print(explain_topic("Neural Networks"))
print(explain_topic("Python"))

=== Task 1 Output ===
Explain Machine Learning in simple terms for beginners
Explain Neural Networks in simple terms for beginners
Explain Python in simple terms for beginners


---
## Task 2: Multi-Input Prompt System (15%)

**Goal:** Build a template with three inputs — `topic`, `audience`, and `tone`.

In [4]:
# Define a multi-input PromptTemplate
task2_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

# Test cases as specified in the assignment
test_cases = [
    {"topic": "AI",            "audience": "beginners",  "tone": "friendly"},
    {"topic": "Python",        "audience": "kids",        "tone": "fun"},
    {"topic": "Deep Learning", "audience": "engineers",   "tone": "technical"},
]

print("=== Task 2 Output ===")
for tc in test_cases:
    # Use the template's format method — no f-strings
    prompt = task2_template.format(**tc)
    print(f"Input  → topic: '{tc['topic']}' | audience: '{tc['audience']}' | tone: '{tc['tone']}'")
    print(f"Output → {prompt}")
    print()

=== Task 2 Output ===
Input  → topic: 'AI' | audience: 'beginners' | tone: 'friendly'
Output → Explain AI for beginners in a friendly tone

Input  → topic: 'Python' | audience: 'kids' | tone: 'fun'
Output → Explain Python for kids in a fun tone

Input  → topic: 'Deep Learning' | audience: 'engineers' | tone: 'technical'
Output → Explain Deep Learning for engineers in a technical tone



---
## Task 3: Prompt Variations Engine (15%)

**Goal:** Create 3 distinct `PromptTemplate` objects for different styles — Teaching, Interview, and Storytelling.

In [5]:
# Three separate PromptTemplates for different styles
teaching_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} clearly step by step"
)

interview_template = PromptTemplate(
    input_variables=["topic"],
    template="Ask 3 questions about {topic}"
)

storytelling_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} as a story"
)

# Store all variations in a dictionary for easy access
prompt_variations = {
    "Teaching":     teaching_template,
    "Interview":    interview_template,
    "Storytelling": storytelling_template,
}

# Run all three templates for the given topic
topic = "Machine Learning"

print("=== Task 3 Output ===")
print(f"Topic: {topic}\n")
for style, template in prompt_variations.items():
    prompt = template.format(topic=topic)
    print(f"[{style}] → {prompt}")

=== Task 3 Output ===
Topic: Machine Learning

[Teaching] → Explain Machine Learning clearly step by step
[Interview] → Ask 3 questions about Machine Learning
[Storytelling] → Explain Machine Learning as a story


---
## Task 4: ChatPromptTemplate System (15%)

**Goal:** Build a chat-based prompt system using System and User roles, with dynamic role selection.

In [6]:
# System message templates for each role — logic kept separate from template
role_system_messages = {
    "teacher":     "You are an expert teacher. Explain concepts clearly and in a structured way.",
    "interviewer": "You are a professional interviewer. Ask insightful questions about the topic.",
    "motivator":   "You are an enthusiastic motivator. Inspire the user to learn about the topic.",
}

def build_chat_prompt(role, topic):
    """
    Builds a ChatPromptTemplate for the given role and topic.
    
    Parameters:
        role  (str): One of 'teacher', 'interviewer', 'motivator'
        topic (str): The subject to discuss
    
    Returns:
        list of formatted chat messages
    """
    # Fetch the system message for the given role
    system_message = role_system_messages.get(role, "You are a helpful assistant.")

    # Build the ChatPromptTemplate with system + human messages
    chat_template = ChatPromptTemplate.from_messages([
        ("system", system_message),
        ("human", "Please help me understand the topic: {topic}")
    ])

    # Format the template with the provided topic
    messages = chat_template.format_messages(topic=topic)
    return messages


# Test cases
chat_test_cases = [
    {"role": "teacher",     "topic": "Neural Networks"},
    {"role": "interviewer", "topic": "Deep Learning"},
    {"role": "motivator",   "topic": "Data Science"},
]

print("=== Task 4 Output ===")
for tc in chat_test_cases:
    messages = build_chat_prompt(tc["role"], tc["topic"])
    print(f"\nRole: {tc['role'].upper()} | Topic: {tc['topic']}")
    for msg in messages:
        print(f"  [{msg.type.upper()}]: {msg.content}")

=== Task 4 Output ===

Role: TEACHER | Topic: Neural Networks
  [SYSTEM]: You are an expert teacher. Explain concepts clearly and in a structured way.
  [HUMAN]: Please help me understand the topic: Neural Networks

Role: INTERVIEWER | Topic: Deep Learning
  [SYSTEM]: You are a professional interviewer. Ask insightful questions about the topic.
  [HUMAN]: Please help me understand the topic: Deep Learning

Role: MOTIVATOR | Topic: Data Science
  [SYSTEM]: You are an enthusiastic motivator. Inspire the user to learn about the topic.
  [HUMAN]: Please help me understand the topic: Data Science


---
## Task 5: Input Validation Layer (10%)

**Goal:** Create a `validate_inputs` function that checks whether `audience` and `tone` are valid, and assigns defaults if not.

In [7]:
# Define valid values — kept as constants, separate from logic
VALID_AUDIENCES = ["beginner", "intermediate", "expert"]
VALID_TONES     = ["formal", "casual", "fun"]

# Default fallback values
DEFAULT_AUDIENCE = "beginner"
DEFAULT_TONE     = "casual"


def validate_inputs(audience, tone):
    """
    Validates audience and tone inputs.
    Assigns default values if inputs are invalid instead of raising an error.

    Parameters:
        audience (str): Target audience level
        tone     (str): Communication tone

    Returns:
        tuple: (validated_audience, validated_tone)
    """
    # Normalize to lowercase for comparison
    audience = audience.strip().lower()
    tone     = tone.strip().lower()

    # Validate audience — assign default if invalid
    if audience not in VALID_AUDIENCES:
        print(f"  [WARNING] Invalid audience '{audience}'. Defaulting to '{DEFAULT_AUDIENCE}'.")
        audience = DEFAULT_AUDIENCE

    # Validate tone — assign default if invalid
    if tone not in VALID_TONES:
        print(f"  [WARNING] Invalid tone '{tone}'. Defaulting to '{DEFAULT_TONE}'.")
        tone = DEFAULT_TONE

    return audience, tone


# Test the validation function
validation_tests = [
    ("beginner",     "formal"),    # Both valid
    ("kids",         "fun"),       # Invalid audience
    ("expert",       "funny"),     # Invalid tone
    ("professional", "sarcastic"), # Both invalid
    ("Intermediate", "Casual"),    # Mixed case — should be normalized
]

print("=== Task 5 Output ===")
for audience, tone in validation_tests:
    print(f"\nInput  → audience: '{audience}' | tone: '{tone}'")
    validated_audience, validated_tone = validate_inputs(audience, tone)
    print(f"Result → audience: '{validated_audience}' | tone: '{validated_tone}'")

=== Task 5 Output ===

Input  → audience: 'beginner' | tone: 'formal'
Result → audience: 'beginner' | tone: 'formal'

Input  → audience: 'kids' | tone: 'fun'
  [WARNING] Invalid audience 'kids'. Defaulting to 'beginner'.
Result → audience: 'beginner' | tone: 'fun'

Input  → audience: 'expert' | tone: 'funny'
  [WARNING] Invalid tone 'funny'. Defaulting to 'casual'.
Result → audience: 'expert' | tone: 'casual'

Input  → audience: 'professional' | tone: 'sarcastic'
  [WARNING] Invalid audience 'professional'. Defaulting to 'beginner'.
  [WARNING] Invalid tone 'sarcastic'. Defaulting to 'casual'.
Result → audience: 'beginner' | tone: 'casual'

Input  → audience: 'Intermediate' | tone: 'Casual'
Result → audience: 'intermediate' | tone: 'casual'


---
## Task 6: Prompt Generator App (15%)

**Goal:** Build a `generate_prompt()` function that combines validation, style selection, and `PromptTemplate` to produce a structured output.

In [8]:
# Style-specific PromptTemplates — modular and reusable
style_templates = {
    "teaching": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} for {audience} in a {tone} teaching style"
    ),
    "interview": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Ask 3 interview questions about {topic} for {audience} in a {tone} tone"
    ),
    "storytelling": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} for {audience} in a {tone} storytelling style"
    ),
}

# Valid style options
VALID_STYLES = list(style_templates.keys())
DEFAULT_STYLE = "teaching"


def generate_prompt(topic, audience, tone, style):
    """
    Full prompt generation pipeline:
    User Input → Validation → Template Selection → Formatted Prompt → Output

    Parameters:
        topic    (str): Subject to explain
        audience (str): Target audience (beginner/intermediate/expert)
        tone     (str): Communication tone (formal/casual/fun)
        style    (str): Prompt style (teaching/interview/storytelling)

    Returns:
        str: Fully formatted prompt string
    """
    # Step 1 — Validate audience and tone
    audience, tone = validate_inputs(audience, tone)

    # Step 2 — Validate style; assign default if invalid
    style = style.strip().lower()
    if style not in VALID_STYLES:
        print(f"  [WARNING] Invalid style '{style}'. Defaulting to '{DEFAULT_STYLE}'.")
        style = DEFAULT_STYLE

    # Step 3 — Select the appropriate PromptTemplate
    selected_template = style_templates[style]

    # Step 4 — Format and return the prompt (no f-strings)
    return selected_template.format(topic=topic, audience=audience, tone=tone)


# Test the generator
generator_tests = [
    ("Neural Networks", "beginner",     "fun",    "storytelling"),
    ("Python",          "intermediate", "casual", "teaching"),
    ("Deep Learning",   "expert",       "formal", "interview"),
    ("AI Ethics",       "kids",         "happy",  "storytelling"),  # With invalid inputs
]

print("=== Task 6 Output ===")
for topic, audience, tone, style in generator_tests:
    print(f"\nInputs → topic: '{topic}' | audience: '{audience}' | tone: '{tone}' | style: '{style}'")
    result = generate_prompt(topic, audience, tone, style)
    print(f"Generated Prompt → {result}")

=== Task 6 Output ===

Inputs → topic: 'Neural Networks' | audience: 'beginner' | tone: 'fun' | style: 'storytelling'
Generated Prompt → Explain Neural Networks for beginner in a fun storytelling style

Inputs → topic: 'Python' | audience: 'intermediate' | tone: 'casual' | style: 'teaching'
Generated Prompt → Explain Python for intermediate in a casual teaching style

Inputs → topic: 'Deep Learning' | audience: 'expert' | tone: 'formal' | style: 'interview'
Generated Prompt → Ask 3 interview questions about Deep Learning for expert in a formal tone

Inputs → topic: 'AI Ethics' | audience: 'kids' | tone: 'happy' | style: 'storytelling'
  [WARNING] Invalid audience 'kids'. Defaulting to 'beginner'.
  [WARNING] Invalid tone 'happy'. Defaulting to 'casual'.
Generated Prompt → Explain AI Ethics for beginner in a casual storytelling style


---
## Task 7: Template Reusability Test (10%)

**Goal:** Demonstrate that ONE template can produce 5 different outputs — same structure, different inputs.

In [9]:
# ONE reusable template used for all 5 test cases
reusable_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

# 5 different input sets to demonstrate reusability
reusability_inputs = [
    {"topic": "Artificial Intelligence", "audience": "beginner",     "tone": "fun"},
    {"topic": "Data Science",            "audience": "intermediate", "tone": "casual"},
    {"topic": "Natural Language Processing", "audience": "expert",   "tone": "formal"},
    {"topic": "Computer Vision",         "audience": "beginner",     "tone": "casual"},
    {"topic": "Reinforcement Learning",  "audience": "intermediate", "tone": "fun"},
]

print("=== Task 7 Output — Template Reusability Test ===")
print(f"Template: '{reusable_template.template}'\n")

for i, inputs in enumerate(reusability_inputs, start=1):
    # Same template, different inputs each time
    prompt = reusable_template.format(**inputs)
    print(f"Run {i}: {prompt}")

=== Task 7 Output — Template Reusability Test ===
Template: 'Explain {topic} for {audience} in a {tone} tone'

Run 1: Explain Artificial Intelligence for beginner in a fun tone
Run 2: Explain Data Science for intermediate in a casual tone
Run 3: Explain Natural Language Processing for expert in a formal tone
Run 4: Explain Computer Vision for beginner in a casual tone
Run 5: Explain Reinforcement Learning for intermediate in a fun tone


---
## Complete Pipeline Demo

**Full flow:** User Input → Validation → Prompt Template → Dynamic Prompt Generation → Output

In [10]:
def run_pipeline(topic, audience, tone, style):
    """
    Runs the full Mini Prompt Engine pipeline.

    Steps:
        1. Accept user inputs
        2. Validate inputs
        3. Select appropriate PromptTemplate
        4. Generate and return dynamic prompt
    """
    print("\n" + "="*55)
    print("           MINI PROMPT ENGINE — PIPELINE")
    print("="*55)
    print(f"  Step 1 [INPUT]      → topic='{topic}', audience='{audience}', tone='{tone}', style='{style}'")

    # Step 2: Validation
    validated_audience, validated_tone = validate_inputs(audience, tone)
    style = style.strip().lower() if style.strip().lower() in VALID_STYLES else DEFAULT_STYLE
    print(f"  Step 2 [VALIDATION] → audience='{validated_audience}', tone='{validated_tone}', style='{style}'")

    # Step 3: Template Selection
    selected = style_templates[style]
    print(f"  Step 3 [TEMPLATE]   → '{selected.template}'")

    # Step 4: Generate Prompt
    final_prompt = selected.format(topic=topic, audience=validated_audience, tone=validated_tone)
    print(f"  Step 4 [OUTPUT]     → {final_prompt}")
    print("="*55)

    return final_prompt


# Run the pipeline with example inputs from the assignment
run_pipeline("Neural Networks", "beginner", "fun",    "storytelling")
run_pipeline("Python",          "kids",     "casual", "teaching")      # 'kids' will be replaced
run_pipeline("Deep Learning",   "expert",   "formal", "interview")


           MINI PROMPT ENGINE — PIPELINE
  Step 1 [INPUT]      → topic='Neural Networks', audience='beginner', tone='fun', style='storytelling'
  Step 2 [VALIDATION] → audience='beginner', tone='fun', style='storytelling'
  Step 3 [TEMPLATE]   → 'Explain {topic} for {audience} in a {tone} storytelling style'
  Step 4 [OUTPUT]     → Explain Neural Networks for beginner in a fun storytelling style

           MINI PROMPT ENGINE — PIPELINE
  Step 1 [INPUT]      → topic='Python', audience='kids', tone='casual', style='teaching'
  [WARNING] Invalid audience 'kids'. Defaulting to 'beginner'.
  Step 2 [VALIDATION] → audience='beginner', tone='casual', style='teaching'
  Step 3 [TEMPLATE]   → 'Explain {topic} for {audience} in a {tone} teaching style'
  Step 4 [OUTPUT]     → Explain Python for beginner in a casual teaching style

           MINI PROMPT ENGINE — PIPELINE
  Step 1 [INPUT]      → topic='Deep Learning', audience='expert', tone='formal', style='interview'
  Step 2 [VALIDATION] → a

'Ask 3 interview questions about Deep Learning for expert in a formal tone'

---
## Summary

| Task | Description | Weight |
|------|-------------|--------|
| 1 | Replace Hardcoded Prompts with `PromptTemplate` | 10% |
| 2 | Multi-Input Prompt System (topic, audience, tone) | 15% |
| 3 | Prompt Variations Engine (Teaching / Interview / Storytelling) | 15% |
| 4 | ChatPromptTemplate with System + Human roles | 15% |
| 5 | Input Validation Layer with defaults | 10% |
| 6 | Prompt Generator App with full pipeline | 15% |
| 7 | Template Reusability Test (5 runs, 1 template) | 10% |

**Key Principles Followed:**
- ❌ No f-strings used anywhere
- ❌ No hardcoded prompts
- ✅ `input_variables` used in all templates
- ✅ Logic kept separate from templates
- ✅ Modular and reusable design throughout